# Setup

In [1]:
import mariadb
import pandas as pd
import dataframe_image as dfi
#import os
import subprocess

In [2]:
db_connection = mariadb.connect(
    user='root',
    password='root',
    host='localhost',
    port=3307,
    database='wca_dev',
)

# Import DB

In [6]:
cur = db_connection.cursor()

query_text = """
select table_type, table_name
from information_schema.tables
where table_schema = 'wca_dev'
order by if(table_type = 'VIEW', 0, 1), table_name
"""

cur.execute(query_text)
tables_views = cur.fetchall()

print(len(tables_views))
for table_view in tables_views[:5]:
    print(table_view)

cur.close()

128
('BASE TABLE', 'active_storage_attachments')
('BASE TABLE', 'active_storage_blobs')
('BASE TABLE', 'active_storage_variant_records')
('BASE TABLE', 'archive_phpbb3_forums')
('BASE TABLE', 'archive_phpbb3_posts')


In [4]:
cur = db_connection.cursor()

query_text = "set foreign_key_checks = 0;"
print(query_text, '\n', sep='')
cur.execute(query_text)

for table_view in tables_views:
    query_text = ''
    
    if table_view[0] == 'VIEW':
        query_text = f'DROP VIEW IF EXISTS {table_view[1]};'
    else: # table_view[0] == 'BASE TABLE':
        query_text = f'DROP TABLE IF EXISTS {table_view[1]};'

    print(query_text)
    cur.execute(query_text)

query_text = "set foreign_key_checks = 1;"
print('\n', query_text, sep='')
cur.execute(query_text)

cur.close()

set foreign_key_checks = 0;


set foreign_key_checks = 1;


In [ ]:
sql_import_filepath = r'C:\Users\AlexN\Downloads\wca-developer-database-dump\wca-developer-database-dump.sql'
#sql_import_filepath = r'C:\Users\AlexN\Downloads\___test.sql'

command = [r"C:\Program Files\MariaDB 11.8\bin\mysql.exe",
            r"--defaults-file=C:\Program Files\MariaDB 11.8\data\my.ini",
            r"--user=root",
            r"--password=root",
            r"--database=wca_dev",
            fr"--execute=\. {sql_import_filepath}"]

print("Executing command... ")
subprocess.run(command, shell=True)
print("Done! :D")

#output = subprocess.run(command, shell=True, capture_output=True)
#print(output.args)
#print('--------------------------------------------------')
#print(output.stdout.decode("utf-8"))
#print('--------------------------------------------------')
#print(output.stderr.decode("utf-8"))

Executing command... Done! :D


# Check risultati competizioni

In [41]:
competitions_to_check = '''

'''

competitions_list = []

for comp in competitions_to_check.split("\n"):
    if comp == '':
        continue
    
    link_parts = comp.split("/")

    if link_parts[-1] == 'registrations':
        comp = link_parts[-2]
    else:
        comp = link_parts[-1]
    
    competitions_list.append(comp)
    print(comp)


In [54]:
with open(r"SQL_queries\___Check comps results.sql", "r") as f:
    query_text = f.read()

setting, query_check_comps_og = query_text.split(";\n")

print(setting)

/* This script cannot be executed directly, you need to replace the folloqing parameters
 * to make it work:
@COMP_LIST@
@COMP_LIST_WITH_0@
*/


In [38]:
COMP_LIST = ','.join([f"'{x}'" for x in competitions_list])
COMP_LIST_WITH_0 = ','.join([f"('{x}', 0)" for x in competitions_list])

query_check_comps = query_check_comps_og.replace('@COMP_LIST@', COMP_LIST).replace('@COMP_LIST_WITH_0@', COMP_LIST_WITH_0)

In [39]:
cur = db_connection.cursor()

cur.execute(query_check_comps)
check_comps_table = cur.fetchall()

cur.close()

In [40]:
cols = ('competition_id', 'conteggio', 'res_message')

df_check_comps_table = pd.DataFrame(check_comps_table, columns=cols)
df_check_comps_table = df_check_comps_table.rename_axis(index="Index", axis=0)

display(df_check_comps_table)

print("----- Competizioni con risultati mancanti (lista da copiare):")
for comp in df_check_comps_table[df_check_comps_table['res_message'] != '']['competition_id'].to_list():
    print(comp)

,competition_id,conteggio,res_message
Index,,,
0,HungarianOpen2026,1416,
1,Wuppertal22026,867,
2,BucharestSAC2026,645,
3,DreilandereckOpen2026,541,
4,TenerifeOpen2026,301,
5,VentoNegroGravatai2026,126,
6,CosmoCubeFestMedellin2026,117,
7,RedCrossPizzatober2026,72,


----- Competizioni con risultati mancanti (lista da copiare):


# Statistiche "settimanali"

## Medagliere italiano & Numero di solve italiano

In [42]:
with open(r"SQL_queries\Country medal table in a specific year.sql", "r") as f:
    query_text = f.read()

query_medal_table_og = query_text.split(";\n")[1]

with open(r"SQL_queries\Number of solves by people in one county in a specific year.sql", "r") as f:
    query_text = f.read()

setting, query_solves_og = query_text.split(";\n")

In [43]:
print(setting)

/* This script cannot be executed directly, you need to replace the folloqing parameters
 * to make it work:
@YEAR@
@COUNTRY@
*/


In [44]:
YEAR = "2026"
COUNTRY = "'Italy'"

query_medal_table = query_medal_table_og.replace('@YEAR@', YEAR).replace('@COUNTRY@', COUNTRY)
query_solves = query_solves_og.replace('@YEAR@', YEAR).replace('@COUNTRY@', COUNTRY)

In [45]:
cur = db_connection.cursor()

cur.execute(query_medal_table)
medal_table = cur.fetchall()

cur.execute(query_solves)
solves = cur.fetchall()

cur.close()

In [46]:
cols = ('rankk', 'person_id', 'person_name', 'Ori', 'Argenti', 'Bronzi', 'Totali', 'Gare', 'Medaglie_per_gara', 'rank_ratio')

df_medal_table = pd.DataFrame(medal_table, columns=cols)
df_medal_table = df_medal_table.rename_axis(index="Index", axis=0)

#df_medal_table

In [47]:
cols = ('rankk', 'person_id', 'person_name', 'num_solves', 'num_DNF', 'total_attempts', '|', 'num_competitions', 'num_solves_per_comp', 'rank_solves_comp')

df_solves = pd.DataFrame(solves, columns=cols)
df_solves = df_solves.rename_axis(index="Index", axis=0)

#df_solves

In [48]:
df_medal_table.head(5)

,rankk,person_id,person_name,Ori,Argenti,Bronzi,Totali,Gare,Medaglie_per_gara,rank_ratio
Index,,,,,,,,,,
0,1,2016GAMM01,Vincenzo Maria Gammino,14,15,15,44,25,1.7600,21
1,2,2021MARI04,Paolo Marino,14,14,9,37,12,3.0833,7
2,3,2018LOCA01,Valerio Locatelli,13,9,5,27,9,3.0000,8
3,4,2023CREA02,Christian Crea,12,8,12,32,12,2.6667,12
4,5,2021SALE01,Sofia Saletnich,10,13,10,33,7,4.7143,3


In [49]:
df_solves.head(5)

,rankk,person_id,person_name,num_solves,num_DNF,total_attempts,|,num_competitions,num_solves_per_comp,rank_solves_comp
Index,,,,,,,,,,
0,1,2009PROV01,Matteo Provasi,2239,45,2284,|,42,53.3095,81
1,2,2020TOMA01,Fabian Tomasović,1188,101,1289,|,23,51.6522,89
2,3,2023CREA02,Christian Crea,940,22,962,|,12,78.3333,4
3,4,2024RICC01,Riccardo Fiumefreddo,915,4,919,|,17,53.8235,78
4,5,2021MARI04,Paolo Marino,911,73,984,|,12,75.9167,10


In [50]:
df_medal_table_st = df_medal_table.head(25).style.hide(axis=0)
await dfi.export_async(df_medal_table_st, r"Images\Rank 1.png", dpi=200)

In [51]:
df_medal_table_st = df_medal_table.sort_values("rank_ratio").head(25).style.hide(axis=0)
await dfi.export_async(df_medal_table_st, r"Images\Rank 2.png", dpi=200)

In [52]:
df_solves_st = df_solves.head(25).style.hide(axis=0)
await dfi.export_async(df_solves_st, r"Images\Solve 1.png", dpi=200)

In [53]:
df_solves_st = df_solves.sort_values("rank_solves_comp").head(25).style.hide(axis=0)
await dfi.export_async(df_solves_st, r"Images\Solve 2.png", dpi=200)

# Altro